In [1]:
import sys;sys.path.append('../..')
from abslithist import *

In [6]:
df=pd.read_csv(os.path.expanduser('~/lltk_data/corpora/ecco_tcp/metadata.csv'))
# df=df[df.title.str.lower().str.contains('essay')]
df['decade'] = df.year//10*10
df.decade.value_counts()

decade
1790    560
1780    500
1770    446
1760    403
1710    251
1750    236
1700    184
1720    174
1740    160
1730    149
1800     36
1600      1
0         1
Name: count, dtype: int64

In [7]:
norm_d = get_norm_dict()
# norm_d

In [8]:
def score_freqs(path_json):
    with open(path_json) as f:
        freqs = json.load(f)
    out = []
    for word,count in freqs.items():
        if word in norm_d:
            for n in range(count):
                out.append(norm_d[word])
    return np.mean(out)

In [9]:
folder = os.path.expanduser('~/lltk_data/corpora/ecco_tcp/freqs/')
folder

'/Users/ryan/lltk_data/corpora/ecco_tcp/freqs/'

In [10]:
def score_freqs_folder(folder, total=None):
    out = []
    iterr = tqdm(total=total)
    for root,dirs,files in os.walk(folder):
        for file in files:
            if file.endswith('.json'):
                iterr.update(1)
                path_json = os.path.join(root,file)
                score = score_freqs(path_json)
                out.append({'id':path_json.replace(folder,'').replace('.json',''), 'score':score})
    return pd.DataFrame(out)

In [11]:
df_scores = score_freqs_folder(folder, total=len(df))
df_scores

  0%|          | 0/3101 [00:00<?, ?it/s]

100%|██████████| 3101/3101 [00:10<00:00, 306.90it/s]


,id,score
0,K010090.000,-0.187487
1,K093660.005,-0.396258
2,K069226.000,-0.273936
3,K113952.003,0.065603
4,K020965.000,-0.348150
...,...,...
3096,K062689.000,-0.694741
3097,K065082.001,-0.583083
3098,K040928.000,-0.323413
3099,K089849.002,-0.455309


In [12]:
odf=df_scores.dropna().merge(df, on='id', how='left').set_index('id').sort_values('score',ascending=False)
odf

,score,author,title,year,id_ESTC,id_DocNo,id_TCP,id_GaleDocNo,id_ContentSet,id_ImageSetID,extent,pubplace,publisher,date,notes,decade
id,,,,,,,,,,,,,,,,
K068486.002,1.730886,"Blackwell, Elizabeth, fl. 1737.",A curious herbal: containing five hundred cuts...,1782,T83979,CW109891227,K068486.002,CW3309891554,ECMS,1154800302,"2v.,plates ; 2⁰.",London :,"printed for C. Nourse,",1782.,<NOTE>With an index to each volume.</NOTE><NOT...,1780
K106011.000,1.319727,"Moxon, Elizabeth.",English housewifry: Exemplified in above four ...,1752,T133796,CW108755027,K106011.000,CW3308755027,ECMS,667301600,"168,165-212,[22]p.,plate : ill. ; 12⁰.",Leedes :,printed by James Lister; and sold by the autho...,[1752?],"<NOTE>In this edition, the first line on p. 11...",1750
K130910.000,1.257446,"Taylor, Margaret, Mrs.",Mrs. Taylor's family companion: or the whole a...,1795,T222842,CW109006130,K130910.000,CW3309006130,ECMS,1036000400,"[2],164p. ; 12⁰.",London :,"printed for W. Lane, and sold by all other boo...",[1795?],<NOTE>With a half-title.</NOTE><NOTE>Turned ch...,1790
K068486.001,1.109055,"Blackwell, Elizabeth, fl. 1737.",A curious herbal: containing five hundred cuts...,1782,T83979,CW109891227,K068486.001,CW3309891227,ECMS,1154800301,"2v.,plates ; 2⁰.",London :,"printed for C. Nourse,",1782.,<NOTE>With an index to each volume.</NOTE><NOT...,1780
K111672.001,0.939160,"Barbauld, Mrs. (Anna Letitia), 1743-1825.",Lessons for children: of three years old. ...,1779,T142785,CW120376494,K111672.001,CW3320376494,ECRP,543401501,2v. ; 12⁰.,Dublin :,"printed and sold by R. Jackson,",1779.,<NOTE>Anonymous. By Anna Barbauld.</NOTE><NOTE...,1770
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
K004461.000,-1.069218,"Gerard, Alexander, 1728-1795.",The influence of the pastoral office on the ch...,1760,N7474,CW119664119,K004461.000,CW3319664119,ECRP,126202100,"[2],75,[1]p. ; 8⁰.",Aberdeen :,printed by J. Chalmers; and sold by And. Milla...,1760.,<NOTE>Reproduction of original from the Harvar...,1760
K006305.000,-1.072566,"Defoe, Daniel, 1661?-1731.",The opinion of a known Dissenter on the bill f...,1703,N10604,CW104385285,K006305.000,CW3304385285,ECSS,127102000,[2]p. ; 1/2⁰.,[London :,"printed, and are to be sold by J. Nutt,",1703] [1702?],<NOTE>A known Dissenter = Daniel Defoe.</NOTE>...,1700
K042183.000,-1.073695,"Berkeley, George, 1685-1753.","Passive obedience: or, the Christian doctrine ...",1712,T43742,CW118271822,K042183.000,CW3318271822,ECRP,210101000,"[4],43,[1]p. ; 8⁰.",London :,"printed for H. Clements,",1712.,<NOTE>Reproduction of original from the Britis...,1710


In [13]:
odf.to_pickle('../../data/scores/v3/data.scores.ECCO_TCP.pkl')